In [0]:
%run /Workspace/Users/antoniorad15@gmail.com/ROBOTICS-AI-training-pipeline/pipeline-finetune-gr00t/secrets-template

In [0]:
TMUX_COUNTER = 0

In [0]:
%python
import os

ssh_priv_key = dbutils.secrets.get(scope="brev", key="ssh_private_key")

with open("/tmp/ssh_private_key_4", "w") as f:
    f.write(ssh_priv_key + "\n")
os.chmod("/tmp/ssh_private_key_4", 0o600)

In [0]:
%sh
ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no ubuntu@$BREV_IP << 'EOF'
echo == OS ==
. /etc/os-release
echo $PRETTY_NAME
echo == GPU / Driver ==
nvidia-smi
EOF

In [0]:
%sh
ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no ubuntu@$BREV_IP << 'EOF'
export PATH="$HOME/.local/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin";
curl -LsSf https://astral.sh/uv/install.sh | /bin/sh;
uv --version;
sudo apt-get update
sudo apt-get install -y libglu1-mesa tmux
EOF

In [0]:
%sh
ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no ubuntu@$BREV_IP << 'EOF'
git clone https://github.com/REBELDOT-SOLUTIONS-S-R-L/ROBOTICS-lehome-challenge.git
EOF

In [0]:
%sh
ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no ubuntu@$BREV_IP << 'EOF'
cd ROBOTICS-lehome-challenge
uv sync
git clone -b lehome-cloth-mimic-compat https://github.com/alex-luci/IsaacLab.git third_party/IsaacLab
git checkout auto-annotation-teleop
git status
git reset --hard ab8230758ed5ffd2901b9ebc38f0e097e944b537
source .venv/bin/activate
Yes | ./third_party/IsaacLab/isaaclab.sh -i none
uv pip install -e ./source/lehome
hf download lehome/asset_challenge --repo-type dataset --local-dir Assets
EOF
scp -r -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no \
  /Volumes/workspace/default/assets/so101_follower_eef.usd \
  ubuntu@$BREV_IP:~/ROBOTICS-lehome-challenge/Assets/robots/lerobot/

In [0]:
TMUX_COUNTER += 1
import os
os.environ["TMUX_COUNTER"] = str(TMUX_COUNTER)

In [0]:
%sh
ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no ubuntu@$BREV_IP << EOF
tmux kill-session -t generate_${TMUX_COUNTER} 2>/dev/null || true
tmux new-session -d -s generate_${TMUX_COUNTER}
tmux send-keys -t generate_${TMUX_COUNTER} 'cd ROBOTICS-lehome-challenge' Enter
tmux send-keys -t generate_${TMUX_COUNTER} 'source .venv/bin/activate' Enter
tmux send-keys -t generate_${TMUX_COUNTER} 'yes Yes | python scripts/mimicgen/generate_dataset.py \
--task LeHome-BiSO101-ManagerBased-Garment-Mimic-v0 \
--garment_name Top_Long_Unseen_0 \
--input_file ~/mimicgen_dataset/annotated_dataset_run_${TMUX_COUNTER}.hdf5 \
--output_file Datasets/hdf5_datasets/4_generated_datasets/generated_dataset_home_pos_test_${TMUX_COUNTER}.hdf5 \
--generation_num_trials 100 \
--device cpu \
--num_envs 1 \
--enable_cameras \
--logging_interval 10 \
--log_success \
--headless' Enter
tmux send-keys -t generate_${TMUX_COUNTER} 'python scripts/mimicgen/leisaac_eef_action_process.py \
	--input_file Datasets/hdf5_datasets/4_generated_datasets/*_failed.hdf5 \
	--output_file Datasets/hdf5_datasets/5_final_generated_datasets/converted_failed_${TMUX_COUNTER}.hdf5 \
	--to_joint \
	--headless' Enter
EOF

In [0]:
TMUX_COUNTER += 1
import os
os.environ["TMUX_COUNTER"] = str(TMUX_COUNTER)

In [0]:
%sh
ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no ubuntu@$BREV_IP << EOF
tmux kill-session -t generate_${TMUX_COUNTER} 2>/dev/null || true
tmux new-session -d -s generate_${TMUX_COUNTER}
tmux send-keys -t generate_${TMUX_COUNTER} 'cd ROBOTICS-lehome-challenge' Enter
tmux send-keys -t generate_${TMUX_COUNTER} 'source .venv/bin/activate' Enter
tmux send-keys -t generate_${TMUX_COUNTER} 'yes Yes | python scripts/mimicgen/generate_dataset.py \
--task LeHome-BiSO101-ManagerBased-Garment-Mimic-v0 \
--garment_name Top_Long_Unseen_0 \
--input_file ~/mimicgen_dataset/annotated_dataset_run_${TMUX_COUNTER}.hdf5 \
--output_file Datasets/hdf5_datasets/4_generated_datasets/generated_dataset_home_pos_test_${TMUX_COUNTER}.hdf5 \
--generation_num_trials 100 \
--device cpu \
--num_envs 1 \
--enable_cameras \
--logging_interval 10 \
--log_success \
--headless' Enter
tmux send-keys -t generate_${TMUX_COUNTER} 'python scripts/mimicgen/leisaac_eef_action_process.py \
	--input_file Datasets/hdf5_datasets/4_generated_datasets/generated_dataset_home_pos_test_2_failed.hdf5 \
	--output_file Datasets/hdf5_datasets/5_final_generated_datasets/test1.hdf5 \
	--to_joint \
	--headless' Enter
EOF

In [0]:
%sh
echo "Waiting for training to complete..."
while true; do
  echo "$(date): Checking status..."
  
  OUTPUT=$(echo '
    echo "=== TMUX SESSIONS ==="
    tmux list-sessions 2>&1 || echo "NO_SESSIONS"
  ' | ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no ubuntu@$BREV_IP 2>&1)
  
  EXIT_CODE=$?
  
  echo "--- SSH Exit Code: $EXIT_CODE ---"
  echo "--- Full Output ---"
  echo "$OUTPUT"
  echo "-------------------"
  
  if [ $EXIT_CODE -ne 0 ]; then
    echo "WARNING: SSH command failed!"
  fi
  
  if echo "$OUTPUT" | grep -q "NO_SESSIONS"; then
    echo "No tmux sessions running. Generation complete!"
    break
  fi
  
  sleep 60
done
echo "Show generated data files"
ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no ubuntu@$BREV_IP << EOF
cd \$HOME/ROBOTICS-lehome-challenge/Datasets/hdf5_datasets/4_generated_datasets/
du -sh .
ls -l
cd \$HOME/ROBOTICS-lehome-challenge/Datasets/hdf5_datasets/5_final_generated_datasets/
du -sh .
ls -l
EOF

In [0]:
%sh
rsync -avz --progress \
  -e "ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no" \
  ubuntu@$BREV_IP:~/ROBOTICS-lehome-challenge/Datasets/hdf5_datasets/4_generated_datasets/*.hdf5 \
  /Volumes/workspace/default/hdf5datasets_lehome_many_clothes/

In [0]:
%sh
rsync -avz --progress \
  -e "ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no" \
  ubuntu@$BREV_IP:~/ROBOTICS-lehome-challenge/Datasets/hdf5_datasets/5_final_generated_datasets/*.hdf5 \
  /Volumes/workspace/default/hdf5datasets_lehome_many_clothes/